# Notebook 1: Data Preprocessing & EDA
## ST7082CEM - Big Data Management and Data Visualisation
### Smaran Luitel | Student ID: 250087

This notebook covers:
- SparkSession initialization
- Data loading and schema inspection
- Exploratory Data Analysis (EDA)
- Null/missing value checks
- Data leakage analysis (`Complain` column)
- Feature engineering (encoding, assembling, scaling)
- Export for downstream notebooks and Tableau

In [1]:
import os
import shutil

# Must be set BEFORE SparkSession is created
os.environ["HADOOP_HOME"] = "C:/winutils"
os.environ["PATH"] = os.environ["PATH"] + ";C:/winutils/bin"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("ChurnPipeline_250087") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"Spark UI: http://localhost:4040")

Spark version: 4.1.1
Spark UI: http://localhost:4040


## 1. Load Data & Schema Inspection

In [2]:
DATA_PATH = "../dataset/Customer-Churn-Records.csv"

df_raw = spark.read.csv(DATA_PATH, header=True, inferSchema=True)
df_raw.printSchema()
df_raw.show(5, truncate=False)
print(f"Shape: ({df_raw.count()} rows, {len(df_raw.columns)} columns)")

root
 |-- RowNumber: integer (nullable = true)
 |-- CustomerId: integer (nullable = true)
 |-- Surname: string (nullable = true)
 |-- CreditScore: integer (nullable = true)
 |-- Geography: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Tenure: integer (nullable = true)
 |-- Balance: double (nullable = true)
 |-- NumOfProducts: integer (nullable = true)
 |-- HasCrCard: integer (nullable = true)
 |-- IsActiveMember: integer (nullable = true)
 |-- EstimatedSalary: double (nullable = true)
 |-- Exited: integer (nullable = true)
 |-- Complain: integer (nullable = true)
 |-- Satisfaction Score: integer (nullable = true)
 |-- Card Type: string (nullable = true)
 |-- Point Earned: integer (nullable = true)

+---------+----------+--------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+--------+------------------+---------+------------+
|RowNumber|CustomerId|Surname |CreditSc

## 2. Column Renaming & Type Audit

Column names with spaces cause errors in PySpark ML pipelines. Rename them upfront.

In [3]:
df = df_raw \
    .withColumnRenamed("Satisfaction Score", "SatisfactionScore") \
    .withColumnRenamed("Card Type", "CardType") \
    .withColumnRenamed("Point Earned", "PointEarned")

print("Renamed columns:")
print(df.columns)
print("\nData types:")
for col_name, dtype in df.dtypes:
    print(f"  {col_name}: {dtype}")

Renamed columns:
['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'Complain', 'SatisfactionScore', 'CardType', 'PointEarned']

Data types:
  RowNumber: int
  CustomerId: int
  Surname: string
  CreditScore: int
  Geography: string
  Gender: string
  Age: int
  Tenure: int
  Balance: double
  NumOfProducts: int
  HasCrCard: int
  IsActiveMember: int
  EstimatedSalary: double
  Exited: int
  Complain: int
  SatisfactionScore: int
  CardType: string
  PointEarned: int


## 3. Data Quality Checks

### 3.1 Null & Missing Value Check

In [4]:
from pyspark.sql.functions import col, isnan, when, count

null_counts = df.select([
    count(when(col(c).isNull() | (col(c).cast("string") == ""), c)).alias(c)
    for c in df.columns
])
print("Null/empty counts per column:")
null_counts.show(vertical=True)

Null/empty counts per column:
-RECORD 0----------------
 RowNumber         | 0   
 CustomerId        | 0   
 Surname           | 0   
 CreditScore       | 0   
 Geography         | 0   
 Gender            | 0   
 Age               | 0   
 Tenure            | 0   
 Balance           | 0   
 NumOfProducts     | 0   
 HasCrCard         | 0   
 IsActiveMember    | 0   
 EstimatedSalary   | 0   
 Exited            | 0   
 Complain          | 0   
 SatisfactionScore | 0   
 CardType          | 0   
 PointEarned       | 0   



### 3.2 Duplicate Row Check

In [5]:
total_rows = df.count()
distinct_rows = df.dropDuplicates().count()
print(f"Total rows:    {total_rows}")
print(f"Distinct rows: {distinct_rows}")
print(f"Duplicates:    {total_rows - distinct_rows}")

Total rows:    10000
Distinct rows: 10000
Duplicates:    0


## 4. Exploratory Data Analysis (EDA)

### 4.1 Descriptive Statistics

In [6]:
df.describe().show()

+-------+------------------+-----------------+-------+-----------------+---------+------+------------------+------------------+-----------------+------------------+-------------------+-------------------+-----------------+------------------+------------------+------------------+--------+------------------+
|summary|         RowNumber|       CustomerId|Surname|      CreditScore|Geography|Gender|               Age|            Tenure|          Balance|     NumOfProducts|          HasCrCard|     IsActiveMember|  EstimatedSalary|            Exited|          Complain| SatisfactionScore|CardType|       PointEarned|
+-------+------------------+-----------------+-------+-----------------+---------+------+------------------+------------------+-----------------+------------------+-------------------+-------------------+-----------------+------------------+------------------+------------------+--------+------------------+
|  count|             10000|            10000|  10000|            10000|    

### 4.2 Class Imbalance Analysis (Target Variable: `Exited`)

In [7]:
churn_dist = df.groupBy("Exited").count() \
    .withColumn("percentage", F.round(F.col("count") / total_rows * 100, 2)) \
    .orderBy("Exited")

print("Target variable distribution:")
churn_dist.show()

churned = df.filter("Exited = 1").count()
not_churned = df.filter("Exited = 0").count()
print(f"Churned:     {churned}  ({churned/total_rows*100:.1f}%)")
print(f"Not churned: {not_churned} ({not_churned/total_rows*100:.1f}%)")
print(f"Imbalance ratio: {not_churned/churned:.2f}:1")

Target variable distribution:
+------+-----+----------+
|Exited|count|percentage|
+------+-----+----------+
|     0| 7962|     79.62|
|     1| 2038|     20.38|
+------+-----+----------+

Churned:     2038  (20.4%)
Not churned: 7962 (79.6%)
Imbalance ratio: 3.91:1


### 4.3 Categorical Feature Value Counts

In [8]:
categorical_cols = ["Geography", "Gender", "CardType", "NumOfProducts", "SatisfactionScore", "HasCrCard", "IsActiveMember"]

for col_name in categorical_cols:
    print(f"\n--- {col_name} ---")
    df.groupBy(col_name).count() \
      .withColumn("pct", F.round(F.col("count") / total_rows * 100, 1)) \
      .orderBy("count", ascending=False) \
      .show()


--- Geography ---
+---------+-----+----+
|Geography|count| pct|
+---------+-----+----+
|   France| 5014|50.1|
|  Germany| 2509|25.1|
|    Spain| 2477|24.8|
+---------+-----+----+


--- Gender ---
+------+-----+----+
|Gender|count| pct|
+------+-----+----+
|  Male| 5457|54.6|
|Female| 4543|45.4|
+------+-----+----+


--- CardType ---
+--------+-----+----+
|CardType|count| pct|
+--------+-----+----+
| DIAMOND| 2507|25.1|
|    GOLD| 2502|25.0|
|  SILVER| 2496|25.0|
|PLATINUM| 2495|25.0|
+--------+-----+----+


--- NumOfProducts ---
+-------------+-----+----+
|NumOfProducts|count| pct|
+-------------+-----+----+
|            1| 5084|50.8|
|            2| 4590|45.9|
|            3|  266| 2.7|
|            4|   60| 0.6|
+-------------+-----+----+


--- SatisfactionScore ---
+-----------------+-----+----+
|SatisfactionScore|count| pct|
+-----------------+-----+----+
|                3| 2042|20.4|
|                2| 2014|20.1|
|                4| 2008|20.1|
|                5| 2004|20.0|
|  

### 4.4 Churn Rate by Categorical Features

In [9]:
for col_name in ["Geography", "Gender", "CardType", "NumOfProducts", "SatisfactionScore", "IsActiveMember"]:
    print(f"\n--- Churn Rate by {col_name} ---")
    df.groupBy(col_name).agg(
        F.count("*").alias("total"),
        F.sum("Exited").alias("churned"),
        F.round(F.mean("Exited") * 100, 2).alias("churn_rate_pct")
    ).orderBy("churn_rate_pct", ascending=False).show()


--- Churn Rate by Geography ---
+---------+-----+-------+--------------+
|Geography|total|churned|churn_rate_pct|
+---------+-----+-------+--------------+
|  Germany| 2509|    814|         32.44|
|    Spain| 2477|    413|         16.67|
|   France| 5014|    811|         16.17|
+---------+-----+-------+--------------+


--- Churn Rate by Gender ---
+------+-----+-------+--------------+
|Gender|total|churned|churn_rate_pct|
+------+-----+-------+--------------+
|Female| 4543|   1139|         25.07|
|  Male| 5457|    899|         16.47|
+------+-----+-------+--------------+


--- Churn Rate by CardType ---
+--------+-----+-------+--------------+
|CardType|total|churned|churn_rate_pct|
+--------+-----+-------+--------------+
| DIAMOND| 2507|    546|         21.78|
|PLATINUM| 2495|    508|         20.36|
|  SILVER| 2496|    502|         20.11|
|    GOLD| 2502|    482|         19.26|
+--------+-----+-------+--------------+


--- Churn Rate by NumOfProducts ---
+-------------+-----+-------+-

### 4.5 Numerical Feature Statistics by Churn Status

In [10]:
num_cols_eda = ["CreditScore", "Age", "Balance", "Tenure", "EstimatedSalary", "PointEarned"]

for num_col in num_cols_eda:
    print(f"\n--- {num_col} by Exited ---")
    df.groupBy("Exited").agg(
        F.round(F.mean(num_col), 2).alias("mean"),
        F.round(F.stddev(num_col), 2).alias("stddev"),
        F.min(num_col).alias("min"),
        F.max(num_col).alias("max")
    ).orderBy("Exited").show()


--- CreditScore by Exited ---
+------+------+------+---+---+
|Exited|  mean|stddev|min|max|
+------+------+------+---+---+
|     0|651.84| 95.65|405|850|
|     1|645.41|100.34|350|850|
+------+------+------+---+---+


--- Age by Exited ---
+------+-----+------+---+---+
|Exited| mean|stddev|min|max|
+------+-----+------+---+---+
|     0|37.41| 10.13| 18| 92|
|     1|44.84|  9.76| 18| 84|
+------+-----+------+---+---+


--- Balance by Exited ---
+------+--------+--------+---+---------+
|Exited|    mean|  stddev|min|      max|
+------+--------+--------+---+---------+
|     0|72742.75|62851.58|0.0| 221532.8|
|     1|91109.48|58346.48|0.0|250898.09|
+------+--------+--------+---+---------+


--- Tenure by Exited ---
+------+----+------+---+---+
|Exited|mean|stddev|min|max|
+------+----+------+---+---+
|     0|5.03|  2.88|  0| 10|
|     1|4.93|  2.94|  0| 10|
+------+----+------+---+---+


--- EstimatedSalary by Exited ---
+------+---------+--------+-----+---------+
|Exited|     mean|  stdd

### 4.6 Age Group Binning

In [11]:
df = df.withColumn("AgeGroup",
    F.when(F.col("Age") < 30, "18-29")
     .when(F.col("Age") < 40, "30-39")
     .when(F.col("Age") < 50, "40-49")
     .when(F.col("Age") < 60, "50-59")
     .otherwise("60+")
)

print("Churn rate by Age Group:")
df.groupBy("AgeGroup").agg(
    F.count("*").alias("count"),
    F.round(F.mean("Exited") * 100, 2).alias("churn_rate_pct")
).orderBy("AgeGroup").show()

Churn rate by Age Group:
+--------+-----+--------------+
|AgeGroup|count|churn_rate_pct|
+--------+-----+--------------+
|   18-29| 1641|          7.56|
|   30-39| 4346|         10.88|
|   40-49| 2618|         30.83|
|   50-59|  869|         56.04|
|     60+|  526|         27.95|
+--------+-----+--------------+



## 5. Data Leakage Analysis: The `Complain` Column

The `Complain` column records whether a customer filed a complaint. On the surface, this may seem like a valid predictor of churn. However, it is critical to investigate the relationship between `Complain` and `Exited` before including it in any model.

**Hypothesis:** If `Complain` was recorded *simultaneously with* or *after* the churn event, it constitutes **target leakage** â€” information unavailable at prediction time â€” and must be excluded from all predictive models.

In [12]:
# Cross-tabulation: Complain vs Exited
print("Cross-tabulation: Complain vs Exited")
df.crosstab("Complain", "Exited").show()

# Pearson correlation
corr_value = df.select(F.corr("Complain", "Exited").alias("pearson_corr")).collect()[0][0]
print(f"Pearson Correlation (Complain vs Exited): {corr_value:.6f}")

# Detailed overlap
complainers = df.filter("Complain = 1").count()
complainers_who_churned = df.filter("Complain = 1 AND Exited = 1").count()
churned_who_complained = df.filter("Exited = 1 AND Complain = 1").count()

print(f"\nTotal complainers: {complainers}")
print(f"Complainers who churned: {complainers_who_churned} ({complainers_who_churned/complainers*100:.1f}%)")
print(f"Churners who complained: {churned_who_complained} ({churned_who_complained/churned*100:.1f}%)")
print(f"\nConclusion: Complain has a {corr_value:.2f} Pearson correlation with Exited.")
print("This is near-perfect target leakage. Complain will be EXCLUDED from all ML models.")

Cross-tabulation: Complain vs Exited
+---------------+----+----+
|Complain_Exited|   0|   1|
+---------------+----+----+
|              0|7952|   4|
|              1|  10|2034|
+---------------+----+----+

Pearson Correlation (Complain vs Exited): 0.995693

Total complainers: 2044
Complainers who churned: 2034 (99.5%)
Churners who complained: 2034 (99.8%)

Conclusion: Complain has a 1.00 Pearson correlation with Exited.
This is near-perfect target leakage. Complain will be EXCLUDED from all ML models.


## 6. Feature Engineering

### 6.1 Drop Non-Predictive & Leakage Columns

- `RowNumber`, `CustomerId`, `Surname`: identifiers with no predictive value
- `Complain`: confirmed target leakage (correlation ~0.99 with `Exited`)

In [13]:
COLS_TO_DROP = ["RowNumber", "CustomerId", "Surname", "Complain", "AgeGroup"]
df_model = df.drop(*COLS_TO_DROP)

print("Columns retained for modeling:")
print(df_model.columns)
print(f"\nTotal features (before encoding): {len(df_model.columns) - 1}  (excluding target 'Exited')")

Columns retained for modeling:
['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'SatisfactionScore', 'CardType', 'PointEarned']

Total features (before encoding): 13  (excluding target 'Exited')


### 6.2 Categorical Encoding & ML Pipeline

PySpark MLlib requires numerical input. We use:
- `StringIndexer`: converts string categories to integer indices
- `OneHotEncoder`: converts indices to binary vectors (avoids ordinal assumption)
- `VectorAssembler`: combines all features into a single feature vector
- `StandardScaler`: normalises features to zero mean, unit variance (required for Logistic Regression and K-Means)

In [14]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

# Categorical columns to encode
cat_cols = ["Geography", "Gender", "CardType"]

# Numeric feature columns
num_cols = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts",
            "HasCrCard", "IsActiveMember", "EstimatedSalary",
            "SatisfactionScore", "PointEarned"]

# Build encoding stages
indexers = [StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep") for c in cat_cols]
encoders = [OneHotEncoder(inputCol=c + "_idx", outputCol=c + "_ohe") for c in cat_cols]

# Assemble all features
assembler_inputs = num_cols + [c + "_ohe" for c in cat_cols]
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features_raw", handleInvalid="keep")

# Scale features
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)

# Build and fit pipeline
prep_pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler])
prep_model = prep_pipeline.fit(df_model)
df_prepared = prep_model.transform(df_model)

print("Pipeline stages:")
for i, stage in enumerate(prep_pipeline.getStages()):
    print(f"  {i}: {stage.__class__.__name__}")

print(f"\nPrepared DataFrame columns: {df_prepared.columns}")
df_prepared.select("features", "Exited").show(3, truncate=False)

Pipeline stages:
  0: StringIndexer
  1: StringIndexer
  2: StringIndexer
  3: OneHotEncoder
  4: OneHotEncoder
  5: OneHotEncoder
  6: VectorAssembler
  7: StandardScaler

Prepared DataFrame columns: ['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'SatisfactionScore', 'CardType', 'PointEarned', 'Geography_idx', 'Gender_idx', 'CardType_idx', 'Geography_ohe', 'Gender_ohe', 'CardType_ohe', 'features_raw', 'features']
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+
|features                                                                                        

## 7. Export

Export a clean CSV for Tableau visualisation. Downstream ML notebooks load the raw dataset directly via `spark.read.csv()` and rebuild their own encoding pipelines, so no intermediate file is needed.

In [15]:
import os
os.makedirs("../exports", exist_ok=True)

# Save clean CSV for Tableau (drop identifiers and leakage column)
tableau_df = df.drop("RowNumber", "CustomerId", "Surname", "Complain")
tableau_df.toPandas().to_csv("../exports/churn_clean_for_tableau.csv", index=False)
print("Saved: ../exports/churn_clean_for_tableau.csv")
print("Rows:", tableau_df.count())

Saved: ../exports/churn_clean_for_tableau.csv
Rows: 10000


## 8. Summary

| Check | Result |
|---|---|
| Total records | 10,000 |
| Null values | None |
| Duplicate rows | None |
| Target imbalance | ~20.4% churned (3.91:1 ratio) |
| Data leakage | `Complain` excluded (Pearson ~0.99 with `Exited`) |
| Features for modeling | 10 numeric + 3 categorical (encoded via OHE) |
| Exports | `churn_clean_for_tableau.csv` (for Tableau) |
